# θ 그룹 크기별 Adaptive Quantum COBYLA 비교

동일한 모델 초기화·데이터·손실 평가 예산으로 `theta_group_size`를 비교합니다.

- `1`: 파라미터별 θ
- `16`, `128`: 그룹별 θ
- `None`: 전체 파라미터가 θ 하나 공유

각 그룹의 파라미터는 +1 확률만 공유하며 방향 샘플은 독립적으로 생성됩니다. 검증 MSE는 최적화에 사용하지 않습니다. Qiskit statevector 시뮬레이션 결과이며 양자 우월성의 증거가 아닙니다.

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
from torch import nn
from modules.optimizers.AdaptiveQuantumDirectionCOBYLA import AdaptiveQuantumDirectionCOBYLA

SEEDS = [11, 23, 37]
GROUP_SIZES = [1, 16, 128, None]
ROUNDS = 4
EVALS_PER_ROUND = 12
torch.set_num_threads(min(4, torch.get_num_threads()))


In [ ]:
def make_data(seed):
    rng = np.random.default_rng(seed)
    x = rng.normal(size=(256, 8)).astype("float32")
    y = (0.7 * np.sin(x[:, 0]) + 0.4 * x[:, 1] ** 2
         - 0.5 * x[:, 2] * x[:, 3] + 0.2 * x[:, 4]
         + rng.normal(0, 0.05, 256)).astype("float32")
    order = rng.permutation(len(x))
    train, valid = order[:192], order[192:]
    return (torch.from_numpy(x[train]), torch.from_numpy(y[train, None]),
            torch.from_numpy(x[valid]), torch.from_numpy(y[valid, None]))

def make_model(seed):
    torch.manual_seed(seed + 1000)
    return nn.Sequential(nn.Linear(8, 16), nn.Tanh(), nn.Linear(16, 1))

def run_one(seed, group_size):
    model = make_model(seed)
    xt, yt, xv, yv = make_data(seed)
    optimizer = AdaptiveQuantumDirectionCOBYLA(
        steps=1, direction_cobyla_evaluations=EVALS_PER_ROUND,
        adaptive_learning_rate=0.1, theta_group_size=group_size,
        cobyla_radius=0.1, seed=seed,
    )
    optimizer.prepare(model)
    calls = 0

    def objective():
        nonlocal calls
        calls += 1
        with torch.no_grad():
            return float(nn.functional.mse_loss(model(xt), yt))

    def mse(x, y):
        with torch.no_grad():
            return float(nn.functional.mse_loss(model(x), y))

    history = [{"round": 0, "train_mse": mse(xt, yt), "val_mse": mse(xv, yv)}]
    start = time.perf_counter()
    for round_index in range(1, ROUNDS + 1):
        before = calls
        info = optimizer.optimize(objective)
        assert calls - before == info["optimization_evaluations"]
        assert calls - before <= EVALS_PER_ROUND
        history.append({"round": round_index, "train_mse": mse(xt, yt),
                        "val_mse": mse(xv, yv)})
    return {"seed": seed, "theta_group_size": group_size,
            "label": "shared" if group_size is None else str(group_size),
            "parameters": sum(p.numel() for p in model.parameters()),
            "theta_count": len(optimizer.sampler.theta),
            "objective_calls": calls,
            "elapsed_seconds": time.perf_counter() - start,
            "history": history}


## 실험 실행

각 시드는 동일한 데이터와 모델 초기화를 사용합니다. 그룹 크기별로 독립적인 최적화기를 생성합니다. Qiskit 회로를 파라미터마다 시뮬레이션하므로 실행 시간이 걸릴 수 있습니다.

In [ ]:
results = []
for seed in SEEDS:
    for group_size in GROUP_SIZES:
        result = run_one(seed, group_size)
        results.append(result)
        print(f"seed={seed}, group={result['label']}, "
              f"theta={result['theta_count']}, "
              f"val_mse={result['history'][-1]['val_mse']:.8f}, "
              f"seconds={result['elapsed_seconds']:.2f}")

df = pd.DataFrame([
    {k: v for k, v in result.items() if k != "history"}
    | {"initial_val_mse": result["history"][0]["val_mse"],
       "final_val_mse": result["history"][-1]["val_mse"],
       "val_mse_reduction": result["history"][0]["val_mse"] - result["history"][-1]["val_mse"]}
    for result in results
])
display(df.sort_values(["seed", "theta_count"]))


## 그룹 크기별 요약 및 시드별 짝 비교

`diff_vs_parameter` = 해당 그룹 최종 검증 MSE − 파라미터별(그룹 크기 1) 최종 검증 MSE. 음수면 해당 그룹 설정이 더 낮은 MSE입니다.

In [ ]:
summary = df.groupby("label", sort=False).agg(
    runs=("seed", "count"),
    theta_count=("theta_count", "first"),
    mean_final_val_mse=("final_val_mse", "mean"),
    mean_val_mse_reduction=("val_mse_reduction", "mean"),
    mean_seconds=("elapsed_seconds", "mean"),
    mean_objective_calls=("objective_calls", "mean"),
).reset_index()
display(summary)

paired = df.pivot(index="seed", columns="label", values="final_val_mse")
diff = paired.subtract(paired["1"], axis=0)
diff.columns = [f"diff_vs_parameter_{label}" for label in diff.columns]
display(diff.reset_index())


## Round별 검증 MSE

동일 시드·Round에서 그룹 크기별 검증 MSE를 비교합니다.

In [ ]:
round_rows = [
    {"seed": result["seed"], "label": result["label"],
     "round": point["round"], "train_mse": point["train_mse"],
     "val_mse": point["val_mse"]}
    for result in results for point in result["history"]
]
round_df = pd.DataFrame(round_rows)
display(round_df.pivot(index=["seed", "round"], columns="label", values="val_mse").reset_index())
display(round_df.groupby(["label", "round"], sort=False)["val_mse"].mean().unstack("label"))


## 해석상 한계

- 시드 3개와 합성 데이터에 대한 탐색적 비교이므로 통계적 우위를 단정할 수 없습니다.
- 그룹별 θ는 방향 확률을 공유하지만 개별 파라미터의 손실 기여도를 추정하지 않습니다.
- 현재 구현은 파라미터마다 Qiskit 회로를 생성하므로 θ 개수가 줄어도 샘플링 시간이 비례해서 감소하지 않을 수 있습니다.
- 측정 시간은 실행 환경에 따라 달라집니다. 출력은 실행 전까지 비어 있습니다.